# Optimización de Hiperparámetros — GridSearchCV con Estrategias Mixtas de Balanceo

Notebook que ejecuta **GridSearchCV independiente** para 5 modelos de clasificación multiclase,
maximizando la precisión con dos estrategias de balanceo:

| Estrategia | Modelos | Descripción |
|:----------:|:-------:|:------------|
| `class_weight='balanced'` | Random Forest, SVM (RBF), Bagging | Pesos inversamente proporcionales a la frecuencia de clase. Se entrenan con **datos reales**. |
| **SMOTE** | Gradient Boosting, AdaBoost | Generación de muestras sintéticas de la clase minoritaria **dentro de cada fold** de CV (sin data leakage). |

> **Referencia SMOTE**: Chawla, N.V., Bowyer, K.W., Hall, L.O. & Kegelmeyer, W.P. — *"SMOTE: Synthetic Minority Over-sampling Technique"* (2002).

> **Nota técnica**: Para los pipelines con SMOTE se usa `imblearn.pipeline.Pipeline` en lugar de `sklearn.pipeline.Pipeline`, ya que este último no soporta transformadores que modifiquen tanto X como y simultáneamente.

## 1. Importaciones y configuración global

In [2]:
import time
import numpy as np
import pandas as pd

# Pipeline de imblearn (soporta SMOTE dentro de CV sin data leakage)
from imblearn.pipeline import Pipeline as ImbPipeline
# Pipeline estándar de sklearn (para modelos sin SMOTE)
from sklearn.pipeline import Pipeline

from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# Modelos
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
    BaggingClassifier,
)
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

# Métricas
from sklearn.metrics import accuracy_score, classification_report

# Configuración global
SEED: int = 42
CV_FOLDS: int = 5
N_JOBS: int = -1  # Usa todos los núcleos disponibles
SCORING: str = 'accuracy'

np.random.seed(SEED)

# Validación cruzada estratificada (mantiene proporciones de clase en cada fold)
cv_strategy = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)

print('Imports OK')

Imports OK


## 2. Carga de datos

Se cargan los datos y se aplica la selección de características definida en el EDA
(eliminación de variables con correlación absoluta > 0.99).

In [3]:
# Carga de datos
train_df = pd.read_csv('data/training.csv', index_col=0)
test_df  = pd.read_csv('data/test.csv', index_col=0)

# Variables a eliminar (gemelas con |correlación| > 0.99)
cols_to_drop = ['bpsmt', 'csuhz', 'gwrec', 'glhls', 'bqwyz']
TARGET = 'class'

# Separar features y target
X_train = train_df.drop(columns=[TARGET] + cols_to_drop)
y_train = train_df[TARGET]

X_test = test_df.drop(columns=[TARGET] + cols_to_drop)
y_test = test_df[TARGET]

print(f'Features tras selección: {X_train.shape[1]} (eliminadas {len(cols_to_drop)})')
print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')
print(f'\nDistribución de clases (train):')
print(y_train.value_counts().sort_index())

Features tras selección: 45 (eliminadas 5)
X_train: (3000, 45), X_test: (2000, 45)

Distribución de clases (train):
class
0    2060
1     921
2      19
Name: count, dtype: int64


## 3. Definición de Pipelines y Rejillas de Hiperparámetros

Cada modelo se define como un pipeline independiente con su rejilla de búsqueda.

**Decisión de diseño sobre balanceo:**
- Los modelos que aceptan `class_weight` (Random Forest, SVM, Bagging) usan `sklearn.pipeline.Pipeline` con datos reales.
- Los modelos que **NO** aceptan `class_weight` (Gradient Boosting, AdaBoost) usan `imblearn.pipeline.Pipeline` con SMOTE integrado.

### 3.1 Random Forest (Estrategia: `class_weight='balanced'` — Datos reales)

In [4]:
# Pipeline con StandardScaler + RandomForest con pesos balanceados
pipeline_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(
        class_weight='balanced',
        random_state=SEED
    ))
])

# Rejilla de hiperparámetros: 3 × 3 × 2 = 18 combinaciones
param_grid_rf = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [None, 10, 20],
    'classifier__max_features': ['sqrt', 'log2'],
}

print(f'Pipeline RF: {pipeline_rf.named_steps.keys()}')
print(f'Combinaciones: {3 * 3 * 2} = 18')

Pipeline RF: dict_keys(['scaler', 'classifier'])
Combinaciones: 18 = 18


### 3.2 Gradient Boosting (Estrategia: SMOTE — Datos sintéticos)

`GradientBoostingClassifier` no soporta `class_weight`. Se usa `ImbPipeline` para
que SMOTE genere muestras sintéticas **dentro de cada fold** de la validación cruzada.

In [5]:
# Pipeline con StandardScaler + SMOTE + GradientBoosting
pipeline_gb = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=SEED)),
    ('classifier', GradientBoostingClassifier(random_state=SEED))
])

# Rejilla de hiperparámetros: 3 × 3 × 3 = 27 combinaciones
param_grid_gb = {
    'classifier__n_estimators': [100, 150, 200],
    'classifier__learning_rate': [0.01, 0.05, 0.1],
    'classifier__max_depth': [3, 4, 5],
}

print(f'Pipeline GB: {list(pipeline_gb.named_steps.keys())}')
print(f'Combinaciones: {3 * 3 * 3} = 27')

Pipeline GB: ['scaler', 'smote', 'classifier']
Combinaciones: 27 = 27


### 3.3 AdaBoost (Estrategia: SMOTE — Datos sintéticos)

`AdaBoostClassifier` no soporta `class_weight` directamente. Se usa `ImbPipeline` con SMOTE.
El estimador base es un `DecisionTreeClassifier(max_depth=1)` (decision stump).

In [6]:
# Pipeline con StandardScaler + SMOTE + AdaBoost
pipeline_ada = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=SEED)),
    ('classifier', AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1),
        random_state=SEED
    ))
])

# Rejilla de hiperparámetros: 3 × 3 = 9 combinaciones
param_grid_ada = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__learning_rate': [0.05, 0.1, 1.0],
}

print(f'Pipeline AdaBoost: {list(pipeline_ada.named_steps.keys())}')
print(f'Combinaciones: {3 * 3} = 9')

Pipeline AdaBoost: ['scaler', 'smote', 'classifier']
Combinaciones: 9 = 9


### 3.4 SVM con Kernel RBF (Estrategia: `class_weight='balanced'` — Datos reales)

`SVC` soporta `class_weight` nativamente. El kernel RBF es ideal para fronteras
de decisión no lineales en espacios de alta dimensionalidad.

In [7]:
# Pipeline con StandardScaler + SVM RBF con pesos balanceados
pipeline_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', SVC(
        kernel='rbf',
        class_weight='balanced',
        random_state=SEED
    ))
])

# Rejilla de hiperparámetros: 3 × 4 = 12 combinaciones
param_grid_svm = {
    'classifier__C': [1, 10, 100],
    'classifier__gamma': ['scale', 'auto', 0.01, 0.1],
}

print(f'Pipeline SVM: {pipeline_svm.named_steps.keys()}')
print(f'Combinaciones: {3 * 4} = 12')

Pipeline SVM: dict_keys(['scaler', 'classifier'])
Combinaciones: 12 = 12


### 3.5 Bagging (Estrategia: `class_weight='balanced'` vía estimador base — Datos reales)

`BaggingClassifier` no expone `class_weight` directamente, pero acepta un estimador
base que sí lo soporte. Se usa `DecisionTreeClassifier(class_weight='balanced')`.

In [8]:
# Pipeline con StandardScaler + Bagging (estimador base con pesos balanceados)
pipeline_bag = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', BaggingClassifier(
        estimator=DecisionTreeClassifier(class_weight='balanced'),
        random_state=SEED
    ))
])

# Rejilla de hiperparámetros: 3 × 3 = 9 combinaciones
param_grid_bag = {
    'classifier__n_estimators': [100, 150, 200],
    'classifier__max_samples': [0.7, 0.8, 1.0],
}

print(f'Pipeline Bagging: {pipeline_bag.named_steps.keys()}')
print(f'Combinaciones: {3 * 3} = 9')

Pipeline Bagging: dict_keys(['scaler', 'classifier'])
Combinaciones: 9 = 9


## 4. Ejecución de GridSearchCV

Se registran los 5 modelos y se ejecuta `GridSearchCV` de forma independiente para cada uno.

**Configuración común:**
- `cv=5` (StratifiedKFold con shuffle)
- `scoring='accuracy'`
- `n_jobs=-1` (paralelización máxima)
- `refit=True` (re-entrena con los mejores parámetros sobre todo el train)

In [9]:
# Registro de modelos: (nombre, pipeline, rejilla, estrategia)
models_config = [
    {
        'name': 'Random Forest',
        'pipeline': pipeline_rf,
        'param_grid': param_grid_rf,
        'strategy': 'class_weight (datos reales)',
    },
    {
        'name': 'Gradient Boosting',
        'pipeline': pipeline_gb,
        'param_grid': param_grid_gb,
        'strategy': 'SMOTE (datos sintéticos)',
    },
    {
        'name': 'AdaBoost',
        'pipeline': pipeline_ada,
        'param_grid': param_grid_ada,
        'strategy': 'SMOTE (datos sintéticos)',
    },
    {
        'name': 'SVM (RBF)',
        'pipeline': pipeline_svm,
        'param_grid': param_grid_svm,
        'strategy': 'class_weight (datos reales)',
    },
    {
        'name': 'Bagging',
        'pipeline': pipeline_bag,
        'param_grid': param_grid_bag,
        'strategy': 'class_weight (datos reales)',
    },
]

print(f'Modelos registrados: {[m["name"] for m in models_config]}')
total_combos = sum(
    np.prod([len(v) for v in m['param_grid'].values()])
    for m in models_config
)
print(f'Total combinaciones (todos los modelos): {total_combos}')
print(f'Total fits (× {CV_FOLDS} folds): {total_combos * CV_FOLDS}')

Modelos registrados: ['Random Forest', 'Gradient Boosting', 'AdaBoost', 'SVM (RBF)', 'Bagging']
Total combinaciones (todos los modelos): 75
Total fits (× 5 folds): 375


In [10]:
# Almacén de resultados
grid_results = []

for config in models_config:
    name = config['name']
    pipeline = config['pipeline']
    param_grid = config['param_grid']
    strategy = config['strategy']

    # Número de combinaciones en la rejilla
    n_combinations = int(np.prod([len(v) for v in param_grid.values()]))

    print(f"\n{'='*70}")
    print(f'  MODELO: {name}')
    print(f'  Estrategia de balanceo: {strategy}')
    print(f'  Combinaciones: {n_combinations} × {CV_FOLDS} folds = {n_combinations * CV_FOLDS} fits')
    print(f"{'='*70}")

    # Configurar GridSearchCV
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=cv_strategy,
        scoring=SCORING,
        n_jobs=N_JOBS,
        verbose=1,
        refit=True,
    )

    # Ejecutar búsqueda
    start_time = time.time()
    grid_search.fit(X_train, y_train)
    elapsed = time.time() - start_time

    # Mostrar resultados del modelo
    print(f'\n  ✅ Mejor score ({SCORING}): {grid_search.best_score_:.4f}')
    print(f'  ⏱  Tiempo total: {elapsed:.1f}s')
    print(f'  📋 Mejores parámetros:')
    for param, value in grid_search.best_params_.items():
        # Eliminar prefijo 'classifier__' para legibilidad
        clean_param = param.replace('classifier__', '')
        print(f'     - {clean_param}: {value}')

    # Almacenar resultados
    grid_results.append({
        'Modelo': name,
        'Estrategia': strategy,
        f'Mejor {SCORING}': grid_search.best_score_,
        'Mejores Parámetros': grid_search.best_params_,
        'Tiempo (s)': round(elapsed, 1),
        'grid_search': grid_search,
    })


  MODELO: Random Forest
  Estrategia de balanceo: class_weight (datos reales)
  Combinaciones: 18 × 5 folds = 90 fits
Fitting 5 folds for each of 18 candidates, totalling 90 fits

  ✅ Mejor score (accuracy): 0.7073
  ⏱  Tiempo total: 26.8s
  📋 Mejores parámetros:
     - max_depth: 10
     - max_features: sqrt
     - n_estimators: 100

  MODELO: Gradient Boosting
  Estrategia de balanceo: SMOTE (datos sintéticos)
  Combinaciones: 27 × 5 folds = 135 fits
Fitting 5 folds for each of 27 candidates, totalling 135 fits

  ✅ Mejor score (accuracy): 0.6907
  ⏱  Tiempo total: 586.8s
  📋 Mejores parámetros:
     - learning_rate: 0.1
     - max_depth: 4
     - n_estimators: 200

  MODELO: AdaBoost
  Estrategia de balanceo: SMOTE (datos sintéticos)
  Combinaciones: 9 × 5 folds = 45 fits
Fitting 5 folds for each of 9 candidates, totalling 45 fits

  ✅ Mejor score (accuracy): 0.5660
  ⏱  Tiempo total: 24.2s
  📋 Mejores parámetros:
     - learning_rate: 1.0
     - n_estimators: 200

  MODELO: SVM (R

## 5. Resumen comparativo

In [11]:
# Tabla resumen (sin el objeto GridSearchCV)
summary_df = pd.DataFrame([
    {k: v for k, v in r.items() if k != 'grid_search'}
    for r in grid_results
]).sort_values(f'Mejor {SCORING}', ascending=False).reset_index(drop=True)

display(summary_df[['Modelo', 'Estrategia', f'Mejor {SCORING}', 'Tiempo (s)']])

best = summary_df.iloc[0]
print(f"\n🏆 Mejor modelo global: {best['Modelo']} "
      f"con {SCORING} = {best[f'Mejor {SCORING}']:.4f}")

,Modelo,Estrategia,Mejor accuracy,Tiempo (s)
0,SVM (RBF),class_weight (datos reales),0.722333,4.0
1,Bagging,class_weight (datos reales),0.707667,64.8
2,Random Forest,class_weight (datos reales),0.707333,26.8
3,Gradient Boosting,SMOTE (datos sintéticos),0.690667,586.8
4,AdaBoost,SMOTE (datos sintéticos),0.566000,24.2



🏆 Mejor modelo global: SVM (RBF) con accuracy = 0.7223


## 6. Detalle de mejores parámetros por modelo

In [12]:
for result in grid_results:
    print(f"\n{'─'*50}")
    print(f"📌 {result['Modelo']} ({result['Estrategia']})")
    print(f"   Score: {result[f'Mejor {SCORING}']:.4f}")
    print(f"   Parámetros óptimos:")
    for param, value in result['Mejores Parámetros'].items():
        clean_param = param.replace('classifier__', '')
        print(f"     • {clean_param}: {value}")


──────────────────────────────────────────────────
📌 Random Forest (class_weight (datos reales))
   Score: 0.7073
   Parámetros óptimos:
     • max_depth: 10
     • max_features: sqrt
     • n_estimators: 100

──────────────────────────────────────────────────
📌 Gradient Boosting (SMOTE (datos sintéticos))
   Score: 0.6907
   Parámetros óptimos:
     • learning_rate: 0.1
     • max_depth: 4
     • n_estimators: 200

──────────────────────────────────────────────────
📌 AdaBoost (SMOTE (datos sintéticos))
   Score: 0.5660
   Parámetros óptimos:
     • learning_rate: 1.0
     • n_estimators: 200

──────────────────────────────────────────────────
📌 SVM (RBF) (class_weight (datos reales))
   Score: 0.7223
   Parámetros óptimos:
     • C: 100
     • gamma: scale

──────────────────────────────────────────────────
📌 Bagging (class_weight (datos reales))
   Score: 0.7077
   Parámetros óptimos:
     • max_samples: 0.8
     • n_estimators: 100
